In [ ]:
# Import your existing types and functions
from pathlib import Path

import pandas as pd
import torch
from js_embedding_vis import write_inlined_config
from muutils.collect_warnings import CollateWarnings

from spd.analysis.embed_vis import AnalysisConfig, coactivation_analysis
from spd.analysis.grouping import (
    CoactivationResults,
    get_coactivations,
)
from spd.data_utils import SparseFeatureDataset
from spd.utils import get_device

In [ ]:
# magic autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
DEVICE = get_device()
torch.set_grad_enabled(False)
print(f"Using device: {DEVICE = }")

In [ ]:
coactivations_tms: CoactivationResults = get_coactivations(
    model_path=Path("../data/tms-decomp/model_40000.pth"),
    dataset_cls=SparseFeatureDataset,
    dataset_kwargs=dict(
        value_range=(0.0, 1.0),
        synced_inputs=None,
    ),
    coactivations_kwargs=dict(
        module_groups=[["linear1", "linear2"]],
    ),
    device=DEVICE,
)

In [ ]:
# Full analysis


df: pd.DataFrame
with CollateWarnings(fmt="({count}x) {filename}:{lineno}\n  {category}: {message}"):
    df, metadata = coactivation_analysis(
        group=coactivations_tms["group_0"],
        config=AnalysisConfig(
            embedding_configs={
                "umap": {"n_neighbors": [8, 16, 32]},
                # "isomap": {"n_neighbors": [8, 16, 32]},
            },
            clustering_configs={
                "kmeans": {"n_clusters": [5, 10]},
                # "agglomerative": {"n_clusters": [5, 10]},
            },
            n_components=3,
        ),
    )

# Print structure
print(metadata.describe())

In [ ]:
df_jsonl = df.to_dict(orient="records")

# df_alive_only.to_json(
# 	Path("temp.jsonl"),
# 	orient="records",
# 	lines=True,
# )

write_inlined_config(
    cfg=dict(
        dataFile=None,
        data=df_jsonl,
        numericalPrefix="embed.umap.n_neighbors-32.ax.",
        defaultColorColumn="feat.class.hclust",
        defaultSelectionColumn="feat.class.hclust",
        hoverColumns=[
            "feat.class.hclust",
            "feat.activation_freq",
        ],
    ),
    out_path=Path("../display/embed_vis.html"),
);